# Phase 2: LLM-based Label Re-derivation (Repo-Style)

This notebook runs **Phase 2** of the Knee MRI Multilabel Classification pipeline.

### Context & Objective:
Per competition notice, only a small subset of training studies carry reliable labels in `train.csv`. The radiology `Report` is the authoritative source of truth.
This notebook calls the LLM extraction engine in `src/label_llm_relabel.py` to:
1. Read each report and extract all 12 binary findings with clinical justifications.
2. Cache JSON responses incrementally to `/kaggle/working/data/processed/llm_raw/` so re-runs never recompute finished studies.
3. Compile cleaned training targets into `train_labels_llm.csv`.
4. Audit disagreements against `train.csv` in `label_audit.csv`.

In [ ]:
# --- Cell 1: Sync Repo (Kaggle Notebook Top Cell) ---
import os, sys, subprocess
from pathlib import Path

REPO_URL = os.environ.get("REPO_URL", "https://github.com/Prashanf/Rsna_KneeAbnormality_Detection.git")
REPO_DIR = "/kaggle/working/repo"

if os.path.exists("/kaggle/working"):
    if not os.path.exists(REPO_DIR):
        print(f"Cloning repo from {REPO_URL}...")
        try:
            subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
        except subprocess.CalledProcessError as e:
            print(f"Git clone failed (check REPO_URL or network): {e}")
    else:
        print("Pulling latest commits...")
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

    if os.path.exists(REPO_DIR):
        commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True).stdout.strip()
        print(f"Repo synced at commit: {commit}")
        if REPO_DIR not in sys.path:
            sys.path.insert(0, REPO_DIR)
else:
    local_root = str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd())
    print(f"Running in local workspace: {local_root}")
    if local_root not in sys.path:
        sys.path.insert(0, local_root)

In [ ]:
# --- Cell 2: Purge module cache & load consolidated imports ---
for mod in list(sys.modules):
    if mod.startswith("src.") or mod.startswith("environment."):
        del sys.modules[mod]

from environment.imports import *
print("Environment imports loaded successfully!")

In [ ]:
# --- Cell 3: Configure API Keys & Paths ---
# On Kaggle: use Add-ons -> Secrets to store ANTHROPIC_API_KEY
try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
    print("Retrieved ANTHROPIC_API_KEY from Kaggle Secrets.")
except Exception:
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if api_key:
        print("Using ANTHROPIC_API_KEY from environment variables.")
    else:
        print("No API key detected. Using 'mock' provider for offline dry-run.")

def resolve_paths():
    kaggle_roots = list(Path('/kaggle/input').glob('*knee*')) + list(Path('/kaggle/input').glob('*rsna*'))
    if kaggle_roots:
        return kaggle_roots[0], Path('/kaggle/working/data/processed')
    for candidate in [Path('data/raw'), Path('../data/raw')]:
        if candidate.exists():
            return candidate.resolve(), candidate.parent.resolve() / 'processed'
    return Path('data/raw'), Path('data/processed')

RAW_DIR, PROCESSED_DIR = resolve_paths()
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f"RAW_DIR: {RAW_DIR}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")

In [ ]:
# --- Cell 4: Check Resumption Progress ---
cache_dir = PROCESSED_DIR / 'llm_raw'
cache_dir.mkdir(parents=True, exist_ok=True)
cached_count = len(list(cache_dir.glob('*.json')))

train_csv_path = RAW_DIR / 'train.csv'
if train_csv_path.exists():
    total_studies = len(pd.read_csv(train_csv_path))
    print(f"Progress: {cached_count} / {total_studies} studies already cached ({cached_count/max(total_studies,1)*100:.1f}%)")
else:
    print(f"Cached studies: {cached_count}")

In [ ]:
# --- Cell 5: Run LLM Relabeling ---
provider = "anthropic" if api_key else "mock"
model_name = "claude-3-5-sonnet-20241022"

labels_df, audit_df = relabel_all_studies(
    raw_dir=RAW_DIR,
    processed_dir=PROCESSED_DIR,
    provider=provider,
    model=model_name,
    api_key=api_key,
    limit=None,  # Set to e.g. 50 for a small initial test run
)

In [ ]:
# --- Cell 6: Visualize Disagreements & Agreement Rates ---
import matplotlib.pyplot as plt

audit_path = PROCESSED_DIR / 'label_audit.csv'
if audit_path.exists():
    audit = pd.read_csv(audit_path)
    
    rates = {}
    for label in TARGET_LABELS:
        orig_col = f"{label}_orig"
        llm_col = f"{label}_llm"
        if orig_col in audit.columns and llm_col in audit.columns:
            valid = audit[orig_col].notna()
            if valid.sum() > 0:
                rate = (audit.loc[valid, orig_col] == audit.loc[valid, llm_col]).mean() * 100
                rates[label] = rate

    plt.figure(figsize=(10, 6))
    plt.barh(list(rates.keys()), list(rates.values()), color='#1f77b4')
    plt.xlabel('Agreement Rate (%)')
    plt.title('Original train.csv vs LLM-Derived Label Agreement')
    plt.xlim(0, 105)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
    
    # Display flagged samples
    flagged = audit[audit['is_flagged'] == True]
    print(f"Total flagged studies with disagreements: {len(flagged)} / {len(audit)}")
    display(flagged.head(10))
else:
    print("label_audit.csv not found.")